# 03: Blocking v2 with word-level TF-IDF + cosine similarity
Name channel and address channel, each retrieved per country by cosine similarity (L2-normalized TF-IDF, sparse dot product). Candidates = union of both channels' top-K.

In [1]:
import os, sys, time
os.environ['TMP']=os.environ['TEMP']='D:/tmp'
sys.path.insert(0,'D:/Amazon_ML_Challenge')
import pandas as pd, numpy as np, scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from src.norm import norm_name, norm_addr
W='D:/Amazon_ML_Challenge/work/'
s1=pd.read_parquet(W+'dev_s1.parquet'); gt=pd.read_parquet(W+'dev_gt.parquet')
D={'S2':pd.read_parquet(W+'dev_s2.parquet').reset_index(drop=True),'S3':pd.read_parquet(W+'dev_s3.parquet').reset_index(drop=True)}
q=s1.sample(10000,random_state=0).reset_index(drop=True)
for d in [q,*D.values()]:
    d['nn']=d.business_name.map(norm_name)
    d['na']=[norm_addr(a,c) for a,c in zip(d.business_address,d.country)]
q[['business_name','nn','business_address','na']].head(4)

,business_name,nn,business_address,na
0,Chiropractic Physicians of Long Bottom,chiropractic physicians of long bottom,"Unit 2, OH, 63300 Sr 124, Long Bottom",unit 2 ohio 63300 sr 124 long bottom
1,Ananya & Partners,ananya partners,"Sr No/Plot No, 270/5, Haveli Pune Uruli-Dewach...",sr no plot no 270 5 haveli pune uruli dewachi ...
2,Trichy Energy,trichy energy,"Plot.No.120, Riviera, Kk Nagar, Trichy, Tiruch...",plot no 120 riviera kk nagar trichy tiruchirap...
3,St. Catholic Church Clinic,st catholic church clinic,"8148 Warren Sharon Road, Masury, OH",8148 warren sharon road masury ohio


## TF-IDF + cosine top-K retrieval
Vectors are L2-normalized, so `Q @ T.T` is the cosine similarity. Top-K per query row; scores are kept because they become matcher features later.

In [2]:
def topk_cosine(Q,T,k,chunk=250):
    Tt=T.T.tocsr(); out=[]
    for i in range(0,Q.shape[0],chunk):
        S=(Q[i:i+chunk]@Tt).tocsr()
        for r in range(S.shape[0]):
            a,b=S.indptr[r],S.indptr[r+1]; ix=S.indices[a:b]; v=S.data[a:b]
            if len(v)>k: p=np.argpartition(-v,k)[:k]; ix,v=ix[p],v[p]
            out.append((ix,v))
    return out

def channel(col,src,k,max_df):
    d=D[src]; res=[None]*len(q)
    for c in q.country.unique():
        qi=np.flatnonzero(q.country.values==c); di=np.flatnonzero(d.country.values==c)
        if len(di)==0: res_c=[(np.array([],int),np.array([]))]*len(qi)
        else:
            vec=TfidfVectorizer(token_pattern=r"\S+",lowercase=False,sublinear_tf=True,max_df=max_df,dtype=np.float32)
            T=vec.fit_transform(d[col].values[di]); Q=vec.transform(q[col].values[qi])
            res_c=[(di[ix],v) for ix,v in topk_cosine(Q,T,k)]
        for j,r in zip(qi,res_c): res[j]=r
    return res

In [3]:
K=20; t=time.time(); R={}
for src in ('S2','S3'):
    R[src]={'name':channel('nn',src,K,0.1),'addr':channel('na',src,K,0.05)}
    print(src,'done',round(time.time()-t),'s')

S2 done 54 s


S3 done 68 s


## Candidate recall (vs ground truth)

In [4]:
g=gt.set_index('source1_entity_id').matched_entity_ids.str.split(',')
def evaluate(src):
    eid=D[src].entity_id.values; hit={'name':0,'addr':0,'union':0}; tot=0; ncand=[]
    for i,qid in enumerate(q.entity_id):
        truth={t for t in g[qid] if t.startswith(src)}
        if not truth: continue
        n={eid[j] for j in R[src]['name'][i][0]}; a={eid[j] for j in R[src]['addr'][i][0]}
        tot+=len(truth); hit['name']+=len(truth&n); hit['addr']+=len(truth&a); hit['union']+=len(truth&(n|a)); ncand.append(len(n|a))
    return {k:round(v/tot,3) for k,v in hit.items()}|{'avg_union_cands':round(np.mean(ncand),1)}
for src in ('S2','S3'): print(src,evaluate(src))

S2 {'name': 0.704, 'addr': 0.92, 'union': 0.977, 'avg_union_cands': np.float64(38.5)}


S3 {'name': 0.752, 'addr': 0.915, 'union': 0.985, 'avg_union_cands': np.float64(38.2)}
